# ShortGPT 复现：7–8B 模型上的 BI 删层（Kaggle 2×T4）

**这个 notebook 做什么**：在完整模型上算每层的 BI（Block Influence，层影响力）→ 删掉 BI 最低的 n 层 → 测困惑度（PPL）、零样本准确率和推理速度，与不删层、从后往前删两种做法对比。

**不做训练**：对应原文主实验里"删层后不恢复"的设置（原文另有恢复实验）。LoRA 恢复是我们的下一步。

**定位**：这是 ShortGPT 方法在 Kaggle 上的迁移实验 / 教学实现，不是原文的严格复现。即使换成 Llama-2-7B，校准数据、评测工具和题目提示格式也都和原文不同，数字不能直接对照，只能比较趋势。

## 开跑前在 Kaggle 右侧设置
1. Accelerator：**GPU T4 x2**
2. Internet：**On**
3. 只有用 Llama-2 时才需要：Add-ons → Secrets，新建 `HF_TOKEN`，值是你在 Hugging Face 的 token（要先在 `meta-llama/Llama-2-7b-hf` 页面接受协议，并等通过）。

## 建议顺序
1. `SMOKE = True` 先跑一遍（约 20–30 分钟，含下载）。这时序列更短、batch 更小，只为验证每一格都能跑通，数字不用看。
2. 改成 `SMOKE = False` 正式跑（粗估 1.5–3 小时，取决于模型和下载速度）。中途断了，在同一个会话里从头重新运行即可：已完成的阶段会从结果文件读取并跳过。换了模型或任何设置，会自动换一个结果文件夹，不会误用旧结果。

## 两段代码留给你写
- **TODO 1**：`bi_from_pair`：由一层的输入和输出算 BI
- **TODO 2**：`drop_layers`：从模型里删掉指定的层

每段后面都有一个自测格。参考答案在最底下，**先自己写，卡住超过 20 分钟再看**。

## 和 ShortGPT 原文的差异（写报告时要说明）
| 项目 | ShortGPT | 这里 |
|---|---|---|
| 校准数据（calibration data） | PG19 | WikiText-2 训练集（下载方便） |
| 评测 | MMLU、CMMLU 等，用 OpenCompass | WikiText-2 PPL + piqa / arc_easy / winogrande 零样本，用 lm-eval |
| 精度 | 未说明，多半是 FP16 / BF16 | FP16（T4 不支持 BF16） |

In [ ]:
# 安装依赖。Kaggle 自带 torch，这里只装 / 升级 transformers、accelerate 和 lm-eval。
!pip install -q -U "transformers>=4.51,<5" accelerate "lm-eval>=0.4.5" datasets

In [ ]:
# ======== 配置：只需要改这一格 ========
MODEL_ID = "Qwen/Qwen3-8B"          # 拿到权限后可改成 "meta-llama/Llama-2-7b-hf"，与 ShortGPT 原文直接对照
SMOKE = True                        # True = 小规模测试；确认没问题后改 False

SEQ_LEN = 512 if SMOKE else 2048    # 校准和 PPL 用的序列长度
N_CALIB = 4 if SMOKE else 64        # 校准序列条数（每条 SEQ_LEN 个 token）
PPL_MAX_WINDOWS = 5 if SMOKE else None   # None = 用完整个 WikiText-2 测试集
LMEVAL_TASKS = ["piqa", "arc_easy", "winogrande"]
LMEVAL_LIMIT = 50 if SMOKE else None     # 每个任务只评测前多少题；None = 全部
LMEVAL_BATCH = 1 if SMOKE else 16

SPEED_IN_LEN, SPEED_OUT_LEN = 512, 128   # 测速：固定输入 512、输出 128 个 token
SPEED_BATCHES = [1] if SMOKE else [1, 8]
SPEED_REPS = 1 if SMOKE else 3

# 要比较的配置。method: none = 不删；bi = 删 BI 最低的 n 层；reverse = 删最后 n 层（简单基线）
N_DROP_MAIN = 9    # Llama-2-7B 删 9/32 层，与 ShortGPT 原文一致；Qwen3-8B 是 9/36 层
CONFIGS = [
    {"name": "full",                  "method": "none",    "n": 0},
    {"name": "bi_4",                  "method": "bi",      "n": 4},
    {"name": f"bi_{N_DROP_MAIN}",      "method": "bi",      "n": N_DROP_MAIN},
    {"name": f"reverse_{N_DROP_MAIN}", "method": "reverse", "n": N_DROP_MAIN},
]
if SMOKE:
    CONFIGS = CONFIGS[:2]

SEED = 0
CALIB_DATA = "Salesforce/wikitext:wikitext-2-raw-v1:train（拼接后按顺序切段）"

In [ ]:
import os, gc, json, math, time
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from datasets import load_dataset
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU 数量:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name} {p.total_memory/2**30:.1f} GiB")

# 只有 gated 模型（Llama-2）需要 token
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(token=token)
    print("已登录 Hugging Face")
except Exception as e:
    print("没有读到 HF_TOKEN（用 Qwen3 不需要）:", type(e).__name__)

# ---- 实验签名：所有会影响结果的设置。签名不同 → 结果放进不同文件夹，绝不混用 ----
import hashlib, importlib.metadata as md
def _ver(pkg):
    try: return md.version(pkg)
    except Exception: return None
try:
    from huggingface_hub import HfApi
    REVISION = HfApi().model_info(MODEL_ID).sha     # 模型仓库的具体版本号
except Exception as e:
    REVISION = None
    print("读不到模型版本号:", type(e).__name__)
SIG = {
    "model": MODEL_ID, "revision": REVISION, "dtype": "float16",
    "calib": CALIB_DATA, "n_calib": N_CALIB, "seq_len": SEQ_LEN,
    "ppl_max_windows": PPL_MAX_WINDOWS,
    "tasks": LMEVAL_TASKS, "limit": LMEVAL_LIMIT, "lmeval_batch": LMEVAL_BATCH,
    "speed": [SPEED_IN_LEN, SPEED_OUT_LEN, SPEED_BATCHES, SPEED_REPS, SEED],
    "configs": CONFIGS,
    "versions": {k: _ver(k) for k in ["torch", "transformers", "accelerate", "lm_eval", "datasets"]},
}
sig_hash = hashlib.md5(json.dumps(SIG, sort_keys=True).encode()).hexdigest()[:8]
RUN_DIR = f"/kaggle/working/{MODEL_ID.split('/')[-1]}{'_smoke' if SMOKE else ''}_{sig_hash}"
os.makedirs(RUN_DIR, exist_ok=True)
sig_path = f"{RUN_DIR}/signature.json"
if os.path.exists(sig_path):
    assert json.load(open(sig_path)) == SIG, "签名文件和当前设置不一致，拒绝复用（不应该发生，检查文件夹）"
else:
    json.dump(SIG, open(sig_path, "w"), indent=1, ensure_ascii=False)
RESULTS_PATH = f"{RUN_DIR}/results.json"
BI_PATH = f"{RUN_DIR}/bi.json"
print("结果文件夹:", RUN_DIR)
print("依赖版本:", SIG["versions"])

T0 = time.time()
def elapsed():
    return f"[已用 {(time.time()-T0)/60:.1f} 分钟]"

## 1. 加载模型

- `torch_dtype=torch.float16`：每个参数 2 字节。8B 模型约 16.4 GB，一张 T4（15 GB）放不下。
- `device_map="auto"`：按层切到两张卡上，前一半层在 cuda:0，后一半在 cuda:1。两张卡**依次**工作，不是同时，所以不会比单卡快，只是放得下。
- `max_memory`：每张卡只放约 11 GiB 权重，留出空间给激活值（activations）和 logits。

**检查 NaN 的原因**：FP16 的数值范围比 BF16 小。Qwen3 原本以 BF16 发布，转成 FP16 后可能溢出成 inf / NaN。这里两条短句的检查只是第一道关，之后 BI 和 PPL 还会各查一次；任何一处出现非有限值都会直接报错停下，不会留下无效结果。

**记录真实设备布局**：`auto` 不保证正好对半分，空间不够时甚至会把部分层放到 CPU（offload，会极慢）。下面会打印每张卡上放了哪些层。

In [ ]:
def load_model():
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        max_memory={0: "11GiB", 1: "11GiB"},
    )
    model.eval()
    return model, tok

def first_device(model):
    return model.model.embed_tokens.weight.device

def free_model():
    global model
    try:
        del model
    except NameError:
        pass
    gc.collect()
    torch.cuda.empty_cache()

t = time.time()
model, tok = load_model()
from collections import Counter
print("设备分布:", dict(Counter(str(v) for v in model.hf_device_map.values())))
assert not any(str(v) in ("cpu", "disk") for v in model.hf_device_map.values()), "有部分模型被放到了 CPU/磁盘，速度会失真，先调整 max_memory"
cfg = model.config
L = cfg.num_hidden_layers
print(f"加载用时 {(time.time()-t)/60:.1f} 分钟；层数 L={L}，hidden={cfg.hidden_size}，"
      f"KV 头={cfg.num_key_value_heads}，参数量={sum(p.numel() for p in model.parameters())/1e9:.2f}B")
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} 已用 {torch.cuda.memory_allocated(i)/2**30:.1f} GiB")

# NaN 检查
with torch.no_grad():
    x = tok(["The capital of France is", "def fibonacci(n):"], return_tensors="pt", padding=True).to(first_device(model))
    logits = model(**x, use_cache=False).logits
if not torch.isfinite(logits).all():
    raise RuntimeError("初始 logits 出现 NaN/inf：FP16 溢出。可以换 Llama-2-7B，或者告诉我报错")
print("初始 logits 全部有限 ✓")

## 2. 校准数据

把 WikiText-2 训练集拼成一长串 token，切成 `N_CALIB` 段，每段 `SEQ_LEN` 个 token。

所有序列等长，**没有填充（padding）**，所以算 BI 时不需要遮罩（mask）。

In [ ]:
def build_calib(tok, n, seq_len):
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(t for t in ds["text"] if t.strip())
    ids = tok(text, return_tensors="pt").input_ids[0]
    n = min(n, ids.numel() // seq_len)
    return [ids[i*seq_len:(i+1)*seq_len].unsqueeze(0) for i in range(n)]

calib = build_calib(tok, N_CALIB, SEQ_LEN)
print(f"校准数据：{len(calib)} 段 × {SEQ_LEN} token")

## 3. TODO 1：由一层的输入和输出算 BI

ShortGPT 的定义（第 i 层）：

$$\text{BI}_i = 1 - \mathbb{E}_{X,t}\; \frac{X_{i,t}^\top X_{i+1,t}}{\|X_{i,t}\|\,\|X_{i+1,t}\|}$$

- $X_{i,t}$：第 i 层**输入**在第 t 个 token 位置的隐藏状态（hidden state），长度为 hidden_size 的向量
- $X_{i+1,t}$：同一位置上第 i 层的**输出**
- $\mathbb{E}_{X,t}$：对所有校准序列、所有 token 位置取平均

**你要写的函数**：`bi_from_pair(x_in, x_out)`
- 输入：两个形状为 `[batch, seq_len, hidden]` 的 FP16 张量
- 返回 `(bi_sum, count)`：这一批里所有 token 的 (1 − 余弦相似度) 之和（Python float），以及 token 个数（int）
- 为什么返回"和 + 个数"而不是平均值：要对多批数据累加，最后再相除，才是全体 token 的平均。

**提示**：
1. 先转成 `float32` 再算。FP16 下 4096 维向量的平方和可能溢出。
2. 余弦相似度在哪一维上算？最后结果应该是 `[batch, seq_len]` 的形状。
3. 可以用 `F.cosine_similarity`，也可以自己写点积除以范数。

**为什么不用 `output_hidden_states=True`**：HF 返回的最后一个隐藏状态已经过了最终的 RMSNorm，不是最后一层的原始输出，最后一层的 BI 会算错。所以下面用钩子（hook）直接抓每层的输入和输出。

In [ ]:
# TODO 1（Pan-Yan-0 写，已通过自测）
def bi_from_pair(x_in, x_out):
    # I think it need to judge the dim is equal.
    assert x_in.shape == x_out.shape
    x_in_float32 = x_in.float()
    x_out_float32 = x_out.float()
    cos = F.cosine_similarity(x_in_float32, x_out_float32, dim=-1)
    bi = 1 - cos
    bi_sum = bi.sum().item()
    count = x_in.size(dim=0) * x_in.size(dim=1)
    return bi_sum, count

In [ ]:
# TODO 1 自测：三种已知答案的情况
a = torch.randn(2, 5, 64, dtype=torch.float16)
s, c = bi_from_pair(a, a)
assert c == 10, f"count 应该是 2*5=10，你得到 {c}"
assert abs(s / c - 0.0) < 1e-3, f"输出 = 输入时 BI 应约为 0，你得到 {s/c}"
s, c = bi_from_pair(a, -a)
assert abs(s / c - 2.0) < 1e-3, f"输出 = -输入时 BI 应约为 2，你得到 {s/c}"
s, c = bi_from_pair(a, 3 * a)
assert abs(s / c - 0.0) < 1e-3, "BI 只看方向不看长度：输出 = 3×输入时 BI 应约为 0"
big = torch.full((1, 1, 4096), 300.0, dtype=torch.float16)   # 平方和超过 FP16 上限 65504
s, c = bi_from_pair(big, big)
assert math.isfinite(s) and abs(s) < 1e-3, "大数值时出现溢出：记得先转 float32"
print("TODO 1 通过 ✓")

In [ ]:
# 用钩子收集每层的 BI（只在完整模型上做一次，结果存到 bi.json）
def collect_bi(model, calib):
    layers = model.model.layers
    sums = [0.0] * len(layers)
    counts = [0] * len(layers)
    handles = []
    first_batch, devs = [True], []
    for i, layer in enumerate(layers):
        def hook(module, args, kwargs, output, i=i):
            x_in = args[0] if len(args) > 0 else kwargs["hidden_states"]
            x_out = output[0] if isinstance(output, tuple) else output
            # 两卡交界处：hook 拿到的是 Accelerate 搬运之前的输入，可能还在上一张卡上
            if first_batch[0]:
                devs.append((i, str(x_in.device), str(x_out.device)))
            x_in = x_in.to(x_out.device)
            assert x_in.shape == x_out.shape
            s, c = bi_from_pair(x_in, x_out)
            if not math.isfinite(s):
                raise RuntimeError(f"第 {i} 层的 BI 出现非有限值")
            sums[i] += s
            counts[i] += c
        handles.append(layer.register_forward_hook(hook, with_kwargs=True))
    try:
        with torch.no_grad():
            for k, x in enumerate(calib):
                model(input_ids=x.to(first_device(model)), use_cache=False)
                if first_batch[0]:
                    cross = [d for d in devs if d[1] != d[2]]
                    print("输入和输出不在同一张卡上的层（交界处）:", cross)
                    first_batch[0] = False
                print(f"  校准 {k+1}/{len(calib)}", end="\r")
    finally:
        for h in handles:
            h.remove()   # 一定要移除，否则之后每次前向都会继续算 BI
    return [s / c for s, c in zip(sums, counts)]

if os.path.exists(BI_PATH):
    bi = json.load(open(BI_PATH))["bi"]
    print("从文件读取 BI")
else:
    t = time.time()
    bi = collect_bi(model, calib)
    json.dump({"bi": bi}, open(BI_PATH, "w"), indent=1)
    print(f"BI 计算用时 {(time.time()-t)/60:.1f} 分钟", elapsed())

assert len(bi) == L and all(math.isfinite(v) for v in bi), "BI 长度不等于层数或有非有限值"
order = sorted(range(L), key=lambda i: bi[i])
print("BI 从低到高的层号：", order)
for i in range(L):
    print(f"层 {i:2d}  BI={bi[i]:.4f}  " + "█" * int(bi[i] / max(bi) * 40))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.bar(range(L), bi)
plt.xlabel("layer index"); plt.ylabel("BI"); plt.title(f"Block Influence — {MODEL_ID}")
plt.tight_layout(); plt.savefig(f"{RUN_DIR}/bi.png", dpi=120); plt.show()

## 4. TODO 2：删层

**你要写的函数**：`drop_layers(model, layers_to_drop)`，原地修改并返回 model。要做四件事：

1. `model.model.layers` 是一个 `nn.ModuleList`。用保留的层**按原顺序**组成一个新的 `nn.ModuleList`，替换回去。
2. 更新 `model.config.num_hidden_layers`。
3. 给保留的层**重新编号**：每层的 `self_attn.layer_idx` 改成新位置 0, 1, 2, …
   - 原因：生成时 KV cache 按 `layer_idx` 存取。删掉第 3 层后，原来的第 4 层如果还叫 4，就会访问不存在的位置，或者和别的层冲突。
4. 如果 `model.config` 有 `layer_types` 这个列表（较新的 transformers 有，记录每层是全注意力还是滑动窗口注意力），只保留对应的项。

**不要**删 `embed_tokens`、最后的 `norm` 和 `lm_head`，它们不属于 `layers`。

下面的自测用一个随机初始化的迷你模型（结构相同，只有 6 层、宽 128），在 CPU 上几秒就能跑完，不会动到上面加载的大模型。

In [ ]:
# TODO 2（Pan-Yan-0 写）
def drop_layers(model, layers_to_drop):
    layers = model.model.layers
    keep_layers = [l for l in range(len(layers)) if l not in layers_to_drop]
    new_layers = nn.ModuleList([layers[i] for i in keep_layers])
    for i, _ in enumerate(new_layers):
        new_layers[i].self_attn.layer_idx = i
    model.config.num_hidden_layers = len(new_layers)

    model.model.layers = new_layers

    if getattr(model.config, "layer_types", None) is not None:
        layer_types_new = [model.config.layer_types[i] for i in keep_layers]
        model.config.layer_types = layer_types_new
    return model

In [ ]:
# TODO 2 自测：迷你模型
tiny_cfg = AutoConfig.from_pretrained(MODEL_ID)
tiny_cfg.num_hidden_layers = 6
tiny_cfg.hidden_size = 128
tiny_cfg.intermediate_size = 256
tiny_cfg.num_attention_heads = 4
tiny_cfg.num_key_value_heads = 2
if hasattr(tiny_cfg, "head_dim"):
    tiny_cfg.head_dim = 32
if getattr(tiny_cfg, "layer_types", None) is not None:
    tiny_cfg.layer_types = tiny_cfg.layer_types[:6]
tiny = AutoModelForCausalLM.from_config(tiny_cfg).eval()

kept_before = [tiny.model.layers[i] for i in [0, 2, 4, 5]]
drop_layers(tiny, [1, 3])
assert len(tiny.model.layers) == 4, "应该剩 4 层"
assert isinstance(tiny.model.layers, torch.nn.ModuleList), "layers 要是 nn.ModuleList"
assert all(a is b for a, b in zip(tiny.model.layers, kept_before)), "保留的层要按原顺序，并且是原来的对象"
assert tiny.config.num_hidden_layers == 4, "config.num_hidden_layers 没更新"
assert [l.self_attn.layer_idx for l in tiny.model.layers] == [0, 1, 2, 3], "layer_idx 没重新编号"
if getattr(tiny.config, "layer_types", None) is not None:
    assert len(tiny.config.layer_types) == 4, "config.layer_types 没裁剪"
with torch.no_grad():
    out = tiny.generate(torch.tensor([[1, 2, 3]]), max_new_tokens=5, do_sample=False, use_cache=True)
assert out.shape[1] == 8
print("TODO 2 通过 ✓（删层后带 KV cache 生成也正常）")
del tiny

## 5. 评测函数

- **PPL（困惑度）**：WikiText-2 测试集切成互不重叠的 `SEQ_LEN` 窗口，PPL = exp(平均每个 token 的交叉熵)。越低越好。
- **零样本准确率**：用 lm-eval，比较每个选项的对数似然（log-likelihood），选最高的。`acc_norm` 按选项长度做了归一化。
- **速度**（入门级的估计，不是线上服务的标准测法）：
  - 输入是**固定种子生成的随机 token**，所有配置用同一批，只代表"固定形状的负载"，和生成质量无关。
  - **TTFT 估计**：调用一次 `generate` 只生成 1 个 token 的端到端时间，约等于 prefill（一次性处理整个输入）的时间。
  - **decode 吞吐估计**：(生成 128 个 token 的总时间 − TTFT 估计) 内每秒生成多少 token。
  - 每次重复的原始时间都会保存下来。
  - **显存峰值**：每个 batch 预热之后才清零重新统计，所以只反映这个 batch 正式测速时的峰值。两张卡分别记录，两数相加不一定等于同一时刻的总峰值。
  - 结论只适用于"HF + 按层切到两张卡"这种部署方式，不能直接外推到 vLLM。
  - 计时前后都调用 `torch.cuda.synchronize`，因为 GPU 是异步执行的，不同步的话测到的只是"把任务交出去"的时间。两张卡都要同步。

In [ ]:
_ppl_ids, _ppl_tok = None, None
def eval_ppl(model, tok):
    global _ppl_ids, _ppl_tok
    if _ppl_ids is None or _ppl_tok != tok.name_or_path:   # 换了分词器就重新分词
        _ppl_tok = tok.name_or_path
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
        _ppl_ids = tok("\n\n".join(ds["text"]), return_tensors="pt").input_ids[0]
    n = _ppl_ids.numel() // SEQ_LEN
    if PPL_MAX_WINDOWS:
        n = min(n, PPL_MAX_WINDOWS)
    nll, cnt = 0.0, 0
    with torch.no_grad():
        for i in range(n):
            x = _ppl_ids[i*SEQ_LEN:(i+1)*SEQ_LEN].unsqueeze(0).to(first_device(model))
            logits = model(input_ids=x, use_cache=False).logits[0, :-1].float()
            y = x[0, 1:].to(logits.device)
            loss = F.cross_entropy(logits, y, reduction="sum").item()
            if not math.isfinite(loss):
                raise RuntimeError(f"PPL 第 {i} 个窗口的 loss 非有限")
            nll += loss
            cnt += y.numel()
            del logits
            print(f"  PPL 窗口 {i+1}/{n}", end="\r")
    return math.exp(nll / cnt)

def eval_tasks(model, tok):
    from lm_eval.models.huggingface import HFLM
    from lm_eval import simple_evaluate
    lm = HFLM(pretrained=model, tokenizer=tok, batch_size=LMEVAL_BATCH, max_length=SEQ_LEN)
    out = simple_evaluate(model=lm, tasks=LMEVAL_TASKS, num_fewshot=0, limit=LMEVAL_LIMIT,
                          bootstrap_iters=0, log_samples=False)
    res = {}
    for t in LMEVAL_TASKS:
        r = out["results"][t]
        res[t] = {k.split(",")[0]: v for k, v in r.items() if k.split(",")[0] in ("acc", "acc_norm")}
    return res

def sync_all():
    for d in range(torch.cuda.device_count()):
        torch.cuda.synchronize(d)

def measure_speed(model, tok):
    res = {}
    dev = first_device(model)
    gen = dict(do_sample=False, pad_token_id=tok.pad_token_id, use_cache=True)
    for b in SPEED_BATCHES:
        g = torch.Generator().manual_seed(SEED + b)      # 固定种子：所有配置用同一批输入
        x = torch.randint(1000, 20000, (b, SPEED_IN_LEN), generator=g).to(dev)
        attn = torch.ones_like(x)
        with torch.no_grad():
            model.generate(x, attention_mask=attn, max_new_tokens=8, min_new_tokens=8, **gen)  # 预热
            sync_all()
            for d in range(torch.cuda.device_count()):
                torch.cuda.reset_peak_memory_stats(d)        # 预热之后才开始统计峰值
            ttft, total = [], []
            for _ in range(SPEED_REPS):
                sync_all(); t = time.perf_counter()
                model.generate(x, attention_mask=attn, max_new_tokens=1, min_new_tokens=1, **gen)
                sync_all(); ttft.append(time.perf_counter() - t)
                sync_all(); t = time.perf_counter()
                model.generate(x, attention_mask=attn, max_new_tokens=SPEED_OUT_LEN, min_new_tokens=SPEED_OUT_LEN, **gen)
                sync_all(); total.append(time.perf_counter() - t)
        tt, to = sorted(ttft)[len(ttft)//2], sorted(total)[len(total)//2]   # 取中位数
        assert to > tt, "总时间不大于 TTFT，计时有问题"
        res[f"bs{b}"] = {
            "ttft_est_s": tt,
            "total_s": to,
            "raw_ttft_s": ttft, "raw_total_s": total,
            "decode_tok_per_s_est": b * (SPEED_OUT_LEN - 1) / (to - tt),
            "peak_mem_GiB": [torch.cuda.max_memory_allocated(d) / 2**30 for d in range(torch.cuda.device_count())],
        }
    return res

## 6. 主循环：每个配置重新加载模型 → 删层 → 评测

- 每个配置都从磁盘重新加载完整模型，保证配置之间互不影响（控制变量）。
- 每完成一个**阶段**（PPL、任务、测速）就写入结果文件。重新运行时跳过已完成的阶段，最后一个阶段失败也不用整组重做。
- 删层是对**全部层**按 BI 排序取最低的 n 层，一次删完（ShortGPT 的做法），不是逐层迭代。

In [ ]:
def layers_for(method, n):
    if method == "none":
        return []
    if method == "bi":
        return sorted(sorted(range(L), key=lambda i: bi[i])[:n])
    if method == "reverse":
        return list(range(L - n, L))
    raise ValueError(method)

results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
STAGES = ["ppl", "tasks", "speed"]
def is_done(name):
    return name in results and all(k in results[name] for k in STAGES)
def save():
    json.dump(results, open(RESULTS_PATH, "w"), indent=1, ensure_ascii=False)

todo = [c for c in CONFIGS if not is_done(c["name"])]
print("已完成：", [c["name"] for c in CONFIGS if is_done(c["name"])], "| 待跑：", [c["name"] for c in todo])

durations = []
for k, c in enumerate(todo):
    t_cfg = time.time()
    drop = layers_for(c["method"], c["n"])
    print(f"\n=== {c['name']}：删 {len(drop)} 层 {drop} === {elapsed()}")

    free_model()
    model, tok = load_model()
    drop_layers(model, drop)
    r = results.setdefault(c["name"], {})
    r["dropped"] = drop
    r["n_layers"] = model.config.num_hidden_layers
    r["params_B"] = sum(p.numel() for p in model.parameters()) / 1e9
    # 删层后每张卡上实际保留了几层（旧的 hf_device_map 还按原层号，不可信）
    r["layers_per_device"] = dict(Counter(str(next(l.parameters()).device) for l in model.model.layers))
    print("  删层后每张卡的层数:", r["layers_per_device"])

    if "ppl" not in r:
        t = time.time(); r["ppl"] = eval_ppl(model, tok); r["t_ppl_min"] = (time.time()-t)/60; save()
    print(f"  PPL = {r['ppl']:.2f}")
    if "tasks" not in r:
        t = time.time(); r["tasks"] = eval_tasks(model, tok); r["t_tasks_min"] = (time.time()-t)/60; save()
    print(f"  任务 = {r['tasks']}")
    if "speed" not in r:
        t = time.time(); r["speed"] = measure_speed(model, tok); r["t_speed_min"] = (time.time()-t)/60; save()
    print(f"  速度 = { {bs: round(v['decode_tok_per_s_est'], 1) for bs, v in r['speed'].items()} } tok/s")

    durations.append(time.time() - t_cfg)
    left = len(todo) - k - 1
    print(f"  本配置 {durations[-1]/60:.1f} 分钟；剩 {left} 个，预计还要 {left*sum(durations)/len(durations)/60:.0f} 分钟")

print("\n全部完成", elapsed())

In [ ]:
import pandas as pd
rows = []
for name, r in results.items():
    row = {"config": name, "layers": r["n_layers"], "params_B": round(r["params_B"], 2), "PPL": round(r["ppl"], 2)}
    for t, m in r["tasks"].items():
        row[t] = round(m.get("acc_norm", m.get("acc")) * 100, 1)
    for bs, s in r["speed"].items():
        row[f"TTFT_{bs}(s)"] = round(s["ttft_est_s"], 3)
        row[f"decode_{bs}(tok/s)"] = round(s["decode_tok_per_s_est"], 1)
    rows.append(row)
df = pd.DataFrame(rows).set_index("config")
if "full" in df.index:   # 相对完整模型的速度比
    for col in [c for c in df.columns if c.startswith("decode_")]:
        df[col.replace("decode_", "speedup_")] = (df[col] / df.loc["full", col]).round(2)
df.to_csv(f"{RUN_DIR}/results_table.csv")
df

## 7. 跑完后回答这几个问题（下次一起讨论）

1. BI 最低的层集中在哪一段（前、中、后）？和 ShortGPT 图里"深层冗余"的结论一致吗？第 0 层和最后一层的 BI 为什么偏高？
2. 删 9 层后，PPL 和零样本准确率分别掉了多少？两者下降幅度一致吗？如果不一致，说明了什么？
3. BI 删层和"从后往前删"哪个好？看 PPL 和看准确率，结论一样吗？
4. 删了约 25% 的层，decode 吞吐提升了多少？接近 1/(1−0.25) ≈ 1.33 倍吗？TTFT 呢？如果不接近，可能是哪些部分没有变（提示：embedding、lm_head、两卡之间的数据传输）。
5. （用 Llama-2-7B 时）你的结果和 ShortGPT 表里的数字能直接比吗？列出所有不同的实验条件。
6. 这套评测（PPL + 三个选择题）没有检验哪些能力？（提示：自由生成、长链推理）

---

## 参考答案（先自己写！）

<details>
<summary>TODO 1：bi_from_pair</summary>

```python
def bi_from_pair(x_in, x_out):
    a = x_in.float()
    b = x_out.float()
    cos = F.cosine_similarity(a, b, dim=-1)      # [batch, seq_len]
    bi = 1.0 - cos
    return bi.sum().item(), bi.numel()
```
</details>

<details>
<summary>TODO 2：drop_layers</summary>

```python
def drop_layers(model, layers_to_drop):
    drop = set(layers_to_drop)
    old = model.model.layers
    keep = [i for i in range(len(old)) if i not in drop]
    model.model.layers = torch.nn.ModuleList([old[i] for i in keep])
    for new_i, layer in enumerate(model.model.layers):
        layer.self_attn.layer_idx = new_i
        if hasattr(layer, "layer_idx"):
            layer.layer_idx = new_i
    model.config.num_hidden_layers = len(keep)
    if getattr(model.config, "layer_types", None) is not None:
        model.config.layer_types = [model.config.layer_types[i] for i in keep]
    return model
```
</details>